# Fine-tune YOLOv8 Eye & Face Detector (`0: eye`, `1: face`)

Notebook dùng chung cho **Google Colab / Kaggle / Local GPU** để fine-tune YOLOv8n/s trên bộ pseudo-label `datasets/yolo_eye/`.

- **Class `0: eye`**: dùng chung cho mắt mở, mắt lim dim và mắt nhắm (giữ trọn mí mắt + lông mày).
- **Class `1: face`**: dùng lấy `face_cx` phân biệt slot trái/phải khi chỉ thấy 1 mắt.
- **Tương thích `ADR-001` & `RULES.md`**: hỗ trợ lật ngang (`fliplr=0.5`), tắt lật dọc (`flipud=0.0`), xuất sẵn `best.pt` và `best.onnx`.

In [ ]:
# (Colab/Kaggle) Cài ultralytics nếu môi trường chưa có
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from ultralytics import YOLO

# --- TÙY CHỌN GOOGLE COLAB (bật True nếu dataset nằm trên Google Drive) ---
USE_GDRIVE = False
if USE_GDRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- CẤU HÌNH TRUNG TÂM ---
CFG = {
    "seed": 42,
    # Tự động dò hoặc điền thẳng đường dẫn tới thư mục yolo_eye (vd '/content/drive/MyDrive/yolo_eye')
    "dataset_dir": None,
    "model_weights": "yolov8n.pt",   # đổi thành 'yolov8s.pt' nếu muốn chính xác hơn
    "project_dir": "runs/yolo_eye",
    "run_name": "yolov8n_eye_face",
    "epochs": 50,
    "patience": 10,
    "imgsz": 320,                     # ảnh DDD gốc 227x227 -> 320 vừa nét vừa rất nhanh
    "batch": 64 if torch.cuda.is_available() else 16,
    "lr0": 0.003,
    "lrf": 0.01,
    "optimizer": "AdamW",
    "workers": 4 if torch.cuda.is_available() else 0,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "onnx_opset": 17,
}

seed_everything(CFG["seed"])
print(
    f"PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | "
    f"Device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
)

In [ ]:
def resolve_dataset_dir(explicit_dir: str | Path | None) -> Path:
    if explicit_dir is not None:
        p = Path(explicit_dir).expanduser().resolve()
        if (p / "images" / "train").exists():
            return p
        raise FileNotFoundError(f"Không thấy images/train trong {p}")

    candidates = [
        Path.cwd() / "datasets" / "yolo_eye",
        Path.cwd().parent / "datasets" / "yolo_eye",
        Path("/content/yolo_eye"),
        Path("/content/datasets/yolo_eye"),
        Path("/kaggle/input/yolo-eye/yolo_eye"),
        Path("/kaggle/input/yolo_eye"),
    ]
    for cand in candidates:
        if (cand / "images" / "train").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Không tự tìm thấy thư mục yolo_eye. Hãy điền đường dẫn vào CFG['dataset_dir']."
    )


DATASET_DIR = resolve_dataset_dir(CFG["dataset_dir"])

# Tạo file data_runtime.yaml với đường dẫn thực tế của máy hiện tại (tránh dính path Windows G:/...)
WORK_DIR = Path.cwd() / "yolo_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML_PATH = WORK_DIR / "data_runtime.yaml"

data_yaml = {
    "path": DATASET_DIR.as_posix(),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "eye", 1: "face"},
}
with DATA_YAML_PATH.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Dataset dir : {DATASET_DIR}")
print(f"Runtime yaml: {DATA_YAML_PATH}")
for split in ("train", "val", "test"):
    imgs = list((DATASET_DIR / "images" / split).glob("*"))
    lbls = list((DATASET_DIR / "labels" / split).glob("*.txt"))
    empty_lbls = sum(1 for p in lbls if p.stat().st_size == 0)
    print(f"  [{split:5s}] images={len(imgs):5d} | labels={len(lbls):5d} | empty_neg={empty_lbls}")

In [ ]:
# Kiểm tra nhanh 8 ảnh ngẫu nhiên từ tập train kèm bounding box pseudo-label
train_imgs = sorted((DATASET_DIR / "images" / "train").glob("*"))
rng = random.Random(CFG["seed"])
samples = rng.sample(train_imgs, k=min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = DATASET_DIR / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
            x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
            color = (0, 230, 0) if cls_id == 0 else (255, 140, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    ax.imshow(img)
    ax.set_title(img_path.stem, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Fine-tune YOLOv8 (lưu ý: không flipud cho mặt tài xế; giữ fliplr=0.5 + erasing mô phỏng che khuất)
model = YOLO(CFG["model_weights"])

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=CFG["epochs"],
    patience=CFG["patience"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    lr0=CFG["lr0"],
    lrf=CFG["lrf"],
    optimizer=CFG["optimizer"],
    device=CFG["device"],
    workers=CFG["workers"],
    project=CFG["project_dir"],
    name=CFG["run_name"],
    exist_ok=True,
    seed=CFG["seed"],
    deterministic=True,
    amp=torch.cuda.is_available(),
    # Augmentation phù hợp góc nhìn trực diện tài xế
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.25,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.3,
    close_mosaic=10,
    erasing=0.15,
)

In [ ]:
# Đánh giá model tốt nhất (best.pt) trên tập test độc lập theo người lái
best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"Best checkpoint: {best_pt}")

best_model = YOLO(str(best_pt))
test_metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=CFG["device"],
    workers=CFG["workers"],
)

print("\n=== TEST METRICS ===")
print(f"mAP50     : {test_metrics.box.map50:.4f}")
print(f"mAP50-95  : {test_metrics.box.map:.4f}")
for idx, name in best_model.names.items():
    p, r, ap50, ap = test_metrics.box.class_result(idx)
    print(f"  Class {idx} ({name:4s}) -> P={p:.4f} | R={r:.4f} | AP50={ap50:.4f} | AP50-95={ap:.4f}")

In [ ]:
# Xem thử dự đoán trên 8 ảnh tập test & xuất ra ONNX để ghép vào pipeline Stage 1
test_imgs = sorted((DATASET_DIR / "images" / "test").glob("*"))
test_samples = random.Random(CFG["seed"]).sample(test_imgs, k=min(8, len(test_imgs)))

preds = best_model.predict(
    source=[str(p) for p in test_samples],
    imgsz=CFG["imgsz"],
    conf=0.25,
    iou=0.45,
    device=CFG["device"],
    verbose=False,
)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, res in zip(axes.flat, preds):
    vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(vis)
    n_eye = int((res.boxes.cls == 0).sum()) if res.boxes is not None else 0
    n_face = int((res.boxes.cls == 1).sum()) if res.boxes is not None else 0
    ax.set_title(f"{Path(res.path).stem} (e={n_eye}, f={n_face})", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Xuất ONNX (tương thích ONNXRuntime / TensorRT)
onnx_path = best_model.export(
    format="onnx",
    imgsz=CFG["imgsz"],
    opset=CFG["onnx_opset"],
    simplify=True,
    dynamic=True,
)
print(f"Exported best.pt  -> {best_pt}")
print(f"Exported best.onnx -> {onnx_path}")